# Random Forest Classification

**Dataset**: MOABB BNCI2014-001 (Motor Imagery)  
**Channels**: 22 channels  
**Sampling rate**: 250 Hz  
**Subject**: 1

---

## Overview

We train a Random Forest classifier on band power features and visualize the confusion matrix and top feature importances.

## What this notebook does

Computes band power features, splits into train/test, standardizes, trains RandomForestClassifier, and extracts feature importances.

## What you should expect to see

- Confusion matrix showing classification performance
- Bar chart of top 20 feature importances (top 10 in green)
- Accuracy printed

## Key parameters

| Parameter | Value |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| n_estimators | 100 |
| test_size | 0.2 |


## 1. Install dependencies


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


## 2. Load MOABB dataset

MOABB downloads data automatically on first use (~44 MB).


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


## 3. Explore the data


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


## 4. Train Random Forest


In [ ]:
from scipy.signal import welch

FS = 250
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

print(f'Feature matrix shape: {features.shape}')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=['left_hand', 'right_hand'])

print(f'Accuracy: {accuracy:.4f}')
print(f'Confusion matrix:\n{cm}')

importances = clf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
top_20 = importances[sorted_idx[:20]]
classes = ['left_hand', 'right_hand']


## 5. Interactive plot

**What to look for:**

- The confusion matrix shows overall classification accuracy
- Feature importances reveal which channels and bands contribute most
- The top 10 features (in green) dominate the classification decision


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = ['green' if i < 10 else 'steelblue' for i in range(20)]

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Confusion Matrix',
    'Top 20 Feature Importances (top 10 in green)'))

fig.add_trace(go.Heatmap(z=cm, x=classes, y=classes, colorscale='Blues',
    text=cm, texttemplate='%{text}', textfont={'size': 16}, name='CM', showscale=True), row=1, col=1)
fig.add_trace(go.Bar(x=[str(i) for i in range(20)], y=top_20, marker_color=colors, name='Importance'), row=2, col=1)

fig.update_xaxes(title_text='Predicted', row=1, col=1)
fig.update_yaxes(title_text='True', row=1, col=1)
fig.update_xaxes(title_text='Feature (sorted by importance)', row=2, col=1)
fig.update_yaxes(title_text='Importance', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='Random Forest Classification')
fig.show()


## What did we learn?

- Random Forest provides both classification and feature importance in a single model
- It is robust to overfitting due to ensemble averaging
- Feature importances help identify the most informative channels and frequency bands
